# Lab 09-03 — RAPTOR tree vs flat retrieval (RAPTOR step 3)

**Track 09 · RAPTOR** — the summary tree that compresses a corpus into navigable levels.

Labs 01 and 02 built the RAPTOR summary tree: lab 01 clustered chunk embeddings with a recursive Gaussian mixture, and lab 02 turned each cluster into an LLM summary node, recursing until one root remained. This lab asks the question that decides whether the tree is worth those LLM calls: does *tree retrieval* find the right chunks, and how does it compare with the flat baseline it replaces?

For each of 3 questions from the test set, we run two retrieval paths — flat cosine over all leaf chunks, and tree walk that embeds the question, compares it against summary nodes at each level, descends only into the matching cluster, and returns the best leaf chunks. Both methods get scored by how many gold-answer words land in their top-1 passage.

```text
3 questions (test.parquet)
  -> BGE-embed question + all N=24 passages
  -> flat: cosine over all leaves, rank, take top-1
  -> tree: walk summary levels via cosine, descend into matching cluster, top-1 leaf
  -> gold-answer word overlap in each top-1
  -> comparison verdict
```

The tree walk lives in `src/tools/raptor.py` — at each level it embeds the question, keeps the `top_k` most similar children, descends, and finally returns the best leaf chunks (`collapse=True`). Each step only compares against a handful of summaries, so a query that matches a broad topic can jump straight to the right cluster without scanning every chunk.


## Setup

This notebook mirrors `src/curriculum/09-raptor/03-tree-vs-flat.py` exactly — the same verified code, split into cells. Two prerequisites must hold before it will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — only the tree *build* step calls the LLM (one summarization call per internal node via `build_tree`). The tree retrieval itself is pure embedding + cosine comparison over summary nodes; no LLM calls happen at query time.
- **The corpus on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` and `Data/corpus/rag-mini-wikipedia/test.parquet` (with `question` and `answer` columns), already fetched by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. From the terminal the lab runs as:

```bash
python src/curriculum/09-raptor/03-tree-vs-flat.py
python src/curriculum/09-raptor/03-tree-vs-flat.py --verify
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# is a no-op safety net for fresh environments):
#   langchain-ollama -> the Ollama chat backend behind llms/ollama.py
#   langchain-huggingface -> sentence-transformer embeddings behind embeddings/bge.py
#   pandas           -> read the rag-mini-wikipedia parquet
#   scikit-learn     -> GaussianMixture clustering inside tools/raptor
%pip install langchain-ollama langchain-huggingface pandas scikit-learn


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory — this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) — then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd  # noqa: E402

from embeddings.bge import BGEEmbedding  # noqa: E402
from llms.ollama import OllamaLLM  # noqa: E402
from tools.raptor import build_tree, retrieve  # noqa: E402


## 1. Configuration

`N_PASSAGES = 24` takes the deterministic head of rag-mini-wikipedia — this is the corpus both methods search. `N_QUESTIONS = 3` keeps the demo fast while still showing a pattern. `MAX_CLUSTER_SIZE = 8` caps how many chunks a summary node covers during tree build; `TOP_K = 4` is how many chunks each method returns per question. `BGE_DEVICE = "cpu"` keeps GPU memory free for Ollama. `PREVIEW = 100` limits how many characters of each top passage print in the demo.


In [ ]:
# 1. Configuration
# --------------------------------------------------------------------------
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 24  # corpus both methods search; tree build costs ~1 LLM call/cluster
N_QUESTIONS = 3  # deterministic head of the test set
TOP_K = 4  # retrieved chunks per method per question
MAX_CLUSTER_SIZE = 8  # max chunks per summary node in the tree
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM
PREVIEW = 100  # characters of each top passage to print


# --------------------------------------------------------------------------


## 2. Load — passages + test questions

`passages.parquet` is a plain table with a `passage` column; `head(n)` keeps the first `n` rows so every run works on the same corpus slice. `test.parquet` has `question` and `answer` columns — yes/no questions from the corpus test set. Both loaders return deterministic heads, so every run compares apples to apples.


In [ ]:
# 2. Load — first N passages + first N test questions of rag-mini-wikipedia
# --------------------------------------------------------------------------
def load_passages(path: Path, n: int) -> list[str]:
    """Return the first ``n`` passage texts (deterministic head)."""
    df = pd.read_parquet(path)
    return [str(text).strip() for text in df.head(n)["passage"].tolist()]


def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` ``{"question", "answer"}`` pairs (deterministic)."""
    df = pd.read_parquet(path)
    return [
        {
            "question": str(row["question"]).strip(),
            "answer": str(row["answer"]).strip(),
        }
        for _, row in df.head(n).iterrows()
    ]


# --------------------------------------------------------------------------


## 3. Experiment — build the tree once, then compare flat vs tree retrieval

This cell contains all the retrieval logic in one place:

- **`_cosine`** — a pure-Python cosine similarity helper used by the flat baseline.
- **`flat_top_k`** — ranks every passage against the question vector using cosine, returns the top `top_k` with their ids and scores. This is the brute-force baseline: O(N) similarity computations, no summaries.
- **`answer_contains`** — normalized substring check: are the gold answer's words in the text?
- **`run_experiment`** — builds the tree via `build_tree` (recursive GMM clustering + LLM summarization, costing one LLM call per internal node), then for each question runs flat retrieval (`flat_top_k` over all leaf passages) and tree retrieval (`tools.raptor.retrieve` which walks summary levels via cosine and descends only into the matching cluster). Both methods return `top_k` passages with ids and scores; each top-1 is checked for gold-answer word overlap.

The retrieval contrast is the core lesson: flat scans every chunk; tree walks summary levels and only descends into the cluster that matches the question.


In [ ]:
# 3. Experiment — build the tree once, then compare flat vs tree retrieval
# --------------------------------------------------------------------------
def _cosine(a: list[float], b: list[float]) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def flat_top_k(
    passages: list[str],
    passage_vecs: list[list[float]],
    question_vec: list[float],
    top_k: int,
) -> dict:
    """Rank all passages against the question; return the ``top_k``."""
    ranked = sorted(
        range(len(passages)),
        key=lambda i: _cosine(question_vec, passage_vecs[i]),
        reverse=True,
    )[:top_k]
    return {
        "texts": [passages[i] for i in ranked],
        "ids": list(ranked),
        "scores": [round(_cosine(question_vec, passage_vecs[i]), 3)
                   for i in ranked],
    }


def answer_contains(gold: str, text: str) -> bool:
    """Normalized substring check: are the gold answer's words in ``text``?"""
    return gold.strip().lower() in text.strip().lower()


def run_experiment() -> dict:
    passages = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, N_QUESTIONS)
    llm = OllamaLLM()  # local qwen2.5-coder:7b; only the tree build uses it
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    t_start = time.perf_counter()
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passages)
    embed_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    tree = build_tree(passages, embedder, llm, max_cluster_size=MAX_CLUSTER_SIZE)
    build_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    rows: list[dict] = []
    for q in questions:
        question_vec = embedder.embed_documents([q["question"]])[0]
        flat = flat_top_k(passages, passage_vecs, question_vec, TOP_K)
        tree_ret = retrieve(
            tree["tree"], q["question"], embedder,
            top_k=TOP_K, collapse=True,
        )
        rows.append(
            {
                "question": q["question"],
                "gold": q["answer"],
                "flat": flat,
                "flat_gold_words": answer_contains(q["answer"], flat["texts"][0]),
                "tree": tree_ret,
                "tree_gold_words": answer_contains(q["answer"], tree_ret["texts"][0]),
            }
        )
    query_s = time.perf_counter() - t0
    total_s = time.perf_counter() - t_start

    return {
        "passages": passages,
        "questions": questions,
        "rows": rows,
        "tree": tree,
        "embed_s": embed_s,
        "build_s": build_s,
        "query_s": query_s,
        "total_s": total_s,
        "agg": {
            "questions": len(rows),
            "flat_gold_words": sum(r["flat_gold_words"] for r in rows),
            "tree_gold_words": sum(r["tree_gold_words"] for r in rows),
        },
    }


# --------------------------------------------------------------------------


## 4. Demo

The demo prints, for each question, the top-1 passage each method surfaces and whether that passage contains the gold answer's words. The verdict line aggregates across all questions. The takeaway: the tree trades a few LLM calls at index time for query-time retrieval that only compares against handfuls of summaries, while the flat baseline stays competitive on small corpora — so the right choice depends on corpus size and query breadth.


In [ ]:
# 4. Demo — print the comparison
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 09-03 — Tree-vs-flat retrieval eval")
    t = exp["tree"]
    print(f"{len(exp['passages'])} passages; tree {t['levels']} levels / "
          f"{t['leaves']} leaves / {t['llm_calls']} LLM calls in "
          f"{exp['build_s']:.1f}s (total {exp['total_s']:.1f}s)")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        flat, tree = row["flat"], row["tree"]
        print(f"\nQ{i}: {row['question']}")
        print(f"    gold answer    : {row['gold']}")
        print(f"    flat  top [id {flat['ids'][0]}] "
              f"{flat['texts'][0][:PREVIEW]}")
        print(f"    flat  gold words : {row['flat_gold_words']}")
        print(f"    tree  top [id {tree['ids'][0]}] "
              f"{tree['texts'][0][:PREVIEW]}")
        print(f"    tree  gold words : {row['tree_gold_words']}")

    a = exp["agg"]
    print(f"\n[5] Gold-answer words surfaced in the top-1 passage "
          f"({a['questions']} questions)")
    print(f"    flat : {a['flat_gold_words']}/{a['questions']}")
    print(f"    tree : {a['tree_gold_words']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    The tree trades a few LLM calls at index time for query-time")
    print("    retrieval that only compares against handfuls of summaries.")
    print("    It shines when a question maps to a whole cluster; the flat")
    print("    baseline stays competitive on small corpora like this one, so")
    print("    the right choice depends on corpus size and query breadth.")


# --------------------------------------------------------------------------


## 5. Verification gate

The gate checks structural correctness of both retrieval paths: valid chunk ids (indices within `0..N`), non-empty texts, `top_k` results per method per question, sane scores in `0.0..1.0`, and a well-formed tree (levels >= 1, leaves == N passages, llm_calls bounded). The gold-answer word hits are reported but not enforced — they are info-only because the test set is small and both methods can legitimately score well.


In [ ]:
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    a = exp["agg"]
    t = exp["tree"]

    for i, row in enumerate(exp["rows"], start=1):
        flat, tree = row["flat"], row["tree"]
        checks.append((f"Q{i} flat returns {TOP_K} non-empty texts + "
                       f"{TOP_K} ids",
                       len(flat["texts"]) == TOP_K
                       and len(flat["ids"]) == TOP_K
                       and all(text.strip() for text in flat["texts"])))
        checks.append((f"Q{i} tree returns {TOP_K} non-empty texts + "
                       f"{TOP_K} ids",
                       len(tree["texts"]) == TOP_K
                       and len(tree["ids"]) == TOP_K
                       and all(text.strip() for text in tree["texts"])))
        checks.append((f"Q{i} flat ids are valid chunk indices (0.."
                       f"{len(exp['passages'])})",
                       all(0 <= idx < len(exp["passages"])
                           for idx in flat["ids"])))
        checks.append((f"Q{i} tree ids are valid chunk indices (0.."
                       f"{len(exp['passages'])})",
                       all(0 <= idx < len(exp["passages"])
                           for idx in tree["ids"])))

    checks.append(("scores are sane (0.0..1.0) for both methods",
                   all(0.0 <= score <= 1.0
                       for row in exp["rows"]
                       for score in row["flat"]["scores"] + row["tree"]["scores"])))
    checks.append((f"tree is well-formed (levels >= 1, leaves == "
                   f"{len(exp['passages'])}, llm_calls <= 40)",
                   t["levels"] >= 1
                   and t["leaves"] == len(exp["passages"])
                   and t["llm_calls"] <= 40))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    print(f"  (info) gold-answer words in top-1: flat "
          f"{a['flat_gold_words']}/{a['questions']}, tree "
          f"{a['tree_gold_words']}/{a['questions']} — reported, not required")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

Tree build plus embeddings take a few minutes depending on hardware. `OllamaLLM()` talks to the local `qwen2.5-coder:7b` server — one LLM call per internal node during tree construction — then the comparison loop embeds each question and runs flat + tree retrieval. No downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Per-question flat-vs-tree top-1 and overlap scores, the verdict line, and the takeaway about when the tree earns its keep.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that both parquet files are intact. The `(info)` line about gold-word hits is reported but not enforced.


In [ ]:
verify_gate(exp)
